In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from typing import Callable, Optional
from utils.algorithms import RGD, PerfGD
from utils.experiment_setup import setup_1d_non_linear_experiment, setup_binary_classification
from poisoning.oracle import oracle_poison_function, RGD_update_estimator, PerfGD_update_estimator, classification_sampling_estimator, gaussian_sampling_estimator
from utils.plotting import plot_results_1d, plot_results_2d

# Non-linear means

In [ ]:
a0 = 1.0
a1 = 1.0

proj_theta = lambda theta: torch.clamp(theta, min=-1.0, max=1.0)
mu, sigma, D_theta, loss, theta_0, grad2_est, f_hat, info = setup_1d_non_linear_experiment(a0=a0, a1=a1, perfGD=True)

## RGD

In [ ]:
n_trials = 10

n = 500
eta = 0.05
max_iter = 30

poison_steps = 40
poison_step_size = 1
epsilon = 1.0
delta = 30

def run_trial():
    _, all_theta_clean, all_losses_clean = RGD(
                D_theta=D_theta,
                loss=loss,
                theta_0=theta_0.clone(),
                proj_theta=proj_theta,
                n=n,
                eta=eta,
                max_iter=max_iter,
                return_losses=True
    )

    _, all_theta_poisoned, all_losses_poisoned = RGD(
                D_theta=D_theta,
                loss=loss,
                theta_0=theta_0.clone(),
                proj_theta=proj_theta,
                n=n,
                eta=eta,
                max_iter=max_iter,
                return_losses=True,
                poison_function=oracle_poison_function,
                theta_update_estimator=RGD_update_estimator,
                sampling_estimator=gaussian_sampling_estimator,
                sampling_estimator_kwargs={'mu': mu, 'sigma': sigma},
                poison_steps=poison_steps,
                poison_step_size=poison_step_size,
                epsilon=epsilon,
                delta=delta,
            )

    return all_theta_clean, all_theta_poisoned, all_losses_clean, all_losses_poisoned

all_theta_clean_list = []
all_theta_poisoned_list = []
all_loss_clean_list = []
all_loss_poisoned_list = []

for trial in range(n_trials):
    all_theta_clean, all_theta_poisoned, all_loss_clean, all_loss_poisoned = run_trial()
    all_theta_clean_list.append(all_theta_clean) 
    all_theta_poisoned_list.append(all_theta_poisoned)
    all_loss_clean_list.append(all_loss_clean)
    all_loss_poisoned_list.append(all_loss_poisoned)

fig, (ax1, ax2) = plot_results_1d(all_theta_clean_list, 
                    all_theta_poisoned_list,
                    all_loss_clean_list, 
                    all_loss_poisoned_list
                    )

plt.show()

### Plot loss and theta gap as a function of delta

In [ ]:
delta_vals = np.linspace(0.0, 100, 100)
epsilon = 0.5
poison_step_size = 1
poison_steps = 40

theta_clean, _, all_losses_clean = RGD(
                D_theta=D_theta,
                loss=loss,
                theta_0=theta_0.clone(),
                proj_theta=proj_theta,
                n=n,
                eta=eta,
                max_iter=max_iter,
                return_losses=True
    )

theta_clean = theta_clean.item()
loss_clean = all_losses_clean[-1]

losses_poisoned, theta_poisoned = [], []
for delta in delta_vals:
    _theta_poisoned, _, all_losses_poisoned = RGD(
                D_theta=D_theta,
                loss=loss,
                theta_0=theta_0.clone(),
                proj_theta=proj_theta,
                n=n,
                eta=eta,
                max_iter=max_iter,
                return_losses=True,
                poison_function=oracle_poison_function,
                theta_update_estimator=RGD_update_estimator,
                sampling_estimator=gaussian_sampling_estimator,
                sampling_estimator_kwargs={'mu': mu, 'sigma': sigma},
                poison_steps=int(float(delta)//poison_step_size) + 1,
                poison_step_size=poison_step_size,
                epsilon=epsilon,
                delta=delta,
    )

    losses_poisoned.append(all_losses_poisoned[-1])
    theta_poisoned.append(_theta_poisoned.item())

losses_poisoned = np.array(losses_poisoned)
theta_poisoned = np.array(theta_poisoned)

fig, ax_left = plt.subplots()
# Left axis: theta difference
ln1 = ax_left.plot(delta_vals, theta_poisoned - theta_clean, color='blue', label='Theta difference')
ax_left.set_xlabel('Delta')
ax_left.set_ylabel('Theta difference', 
                   color='blue')
ax_left.tick_params(axis='y', labelcolor='blue')

# Right axis: loss difference
ax_right = ax_left.twinx()
ln2 = ax_right.plot(delta_vals, losses_poisoned - loss_clean, color='red', label='Loss difference')
ax_right.set_ylabel('Loss difference', color='red')
ax_right.tick_params(axis='y', labelcolor='red')

# Combined legend
lines = ln1 + ln2
labels = [l.get_label() for l in lines]
ax_left.legend(lines, labels, loc='lower right')

ax_left.set_title('Difference in Theta and Loss vs Delta')
plt.show()

## PerfGD

In [ ]:
n_trials = 10

n = 500
eta = 0.05
max_iter = 30

poison_steps = 10
poison_step_size = 0.5
epsilon = 0.5
delta = 2.5

def run_trial():
    _, all_theta_clean, all_losses_clean = PerfGD(
                f_hat=f_hat,
                grad2_est=grad2_est,
                D_theta=D_theta,
                loss=loss,
                theta_0=theta_0.clone(),
                proj_theta=proj_theta,
                n=n,
                eta=eta,
                max_iter=max_iter,
                return_losses=True  
    )

    _, all_theta_poisoned, all_losses_poisoned = PerfGD(
                f_hat=f_hat,
                grad2_est=grad2_est,
                D_theta=D_theta,
                loss=loss,
                theta_0=theta_0.clone(),
                proj_theta=proj_theta,
                n=n,
                eta=eta,
                max_iter=max_iter,
                return_losses=True,
                poison_function=oracle_poison_function,
                theta_update_estimator=PerfGD_update_estimator,
                poison_steps=poison_steps,
                poison_step_size=poison_step_size,
                epsilon=epsilon,
                delta=delta,
                sampling_estimator=gaussian_sampling_estimator,
                sampling_estimator_kwargs={'mu': mu, 'sigma': sigma},
    )

    return all_theta_clean, all_theta_poisoned, all_losses_clean, all_losses_poisoned

all_theta_clean_list = []
all_theta_poisoned_list = []
all_loss_clean_list = []
all_loss_poisoned_list = []

for trial in range(n_trials):
    all_theta_clean, all_theta_poisoned, all_loss_clean, all_loss_poisoned = run_trial()
    all_theta_clean_list.append(all_theta_clean) 
    all_theta_poisoned_list.append(all_theta_poisoned)
    all_loss_clean_list.append(all_loss_clean)
    all_loss_poisoned_list.append(all_loss_poisoned)

fig, (ax1, ax2) = plot_results_1d(all_theta_clean_list, 
                    all_theta_poisoned_list,
                    all_loss_clean_list, 
                    all_loss_poisoned_list
                    )

plt.show()

### Plot theta and loss gap vs. delta

In [ ]:
delta_vals = np.linspace(0.1, 5.0, 100)
poison_step_size = 0.5
poison_steps = 10
epsilon = 0.5

theta_clean, _, all_losses_clean = PerfGD(
                f_hat=f_hat,
                grad2_est=grad2_est,
                D_theta=D_theta,
                loss=loss,
                theta_0=theta_0.clone(),
                proj_theta=proj_theta,
                n=n,
                eta=eta,
                max_iter=max_iter,
                return_losses=True  
    )

theta_clean = theta_clean.item()
loss_clean = all_losses_clean[-1]

losses_poisoned, theta_poisoned = [], []
for delta in delta_vals:
    _theta_poisoned, _, all_losses_poisoned = PerfGD(
                f_hat=f_hat,
                grad2_est=grad2_est,
                D_theta=D_theta,
                loss=loss,
                theta_0=theta_0.clone(),
                proj_theta=proj_theta,
                n=n,
                eta=eta,
                max_iter=max_iter,
                return_losses=True,
                poison_function=oracle_poison_function,
                theta_update_estimator=PerfGD_update_estimator,
                poison_steps=int(float(delta)//poison_step_size) + 1,
                poison_step_size=poison_step_size, 
                sampling_estimator=gaussian_sampling_estimator,
                sampling_estimator_kwargs={'mu': mu, 'sigma': sigma},
                epsilon=epsilon,
                delta=delta,
    )

    losses_poisoned.append(all_losses_poisoned[-1])
    theta_poisoned.append(_theta_poisoned.item())

losses_poisoned = np.array(losses_poisoned)
theta_poisoned = np.array(theta_poisoned)

fig, ax_left = plt.subplots()
# Left axis: theta difference
ln1 = ax_left.plot(delta_vals, theta_poisoned - theta_clean, color='blue', label='Theta difference')
ax_left.set_xlabel('Delta')
ax_left.set_ylabel('Theta difference', 
                   color='blue')
ax_left.tick_params(axis='y', labelcolor='blue')

# Right axis: loss difference
ax_right = ax_left.twinx()
ln2 = ax_right.plot(delta_vals, losses_poisoned - loss_clean, color='red', label='Loss difference')
ax_right.set_ylabel('Loss difference', color='red')
ax_right.tick_params(axis='y', labelcolor='red')

# Combined legend
lines = ln1 + ln2
labels = [l.get_label() for l in lines]
ax_left.legend(lines, labels, loc='lower right')

ax_left.set_title('Difference in Theta and Loss vs Delta')
plt.show()

# Binary Classification

In [ ]:
mu_f, mu_0, sigma_0, sigma_1, D_theta, loss, theta_0, grad2_est, f_hat = setup_binary_classification(perfGD=True)

x, y = D_theta(n=1000, theta=theta_0)

# prettier colors and styled histograms
colors = ['#4C72B0', '#DD8452']  # muted blue and warm orange

x0 = x[(y == 0).bool()].numpy()
x1 = x[(y == 1).bool()].numpy()

plt.hist(x0, bins=30, density=True, alpha=0.65, color=colors[0],
         edgecolor='k', linewidth=0.5, label='Class 0')
plt.hist(x1, bins=30, density=True, alpha=0.65, color=colors[1],
         edgecolor='k', linewidth=0.5, label='Class 1')

plt.xlabel('x')
plt.ylabel('Density')
plt.title('Sample from D(theta)')
plt.legend(frameon=True)
plt.show()

## RGD

In [ ]:
n_trials = 5

n = 500
eta = 0.05
max_iter = 30

poison_steps = 40
poison_step_size = 0.1
epsilon = 0.5
delta = 5

theta_0 = torch.tensor([0.8,1.0])

def run_trial():
    _, all_theta_clean, all_losses_clean = RGD(
                D_theta=D_theta,
                loss=loss,
                theta_0=theta_0.clone(),
                n=n,
                eta=eta,
                max_iter=max_iter,
                return_losses=True
    )

    _, all_theta_poisoned, all_losses_poisoned = RGD(
                D_theta=D_theta,
                loss=loss,
                theta_0=theta_0.clone(),
                n=n,
                eta=eta,
                max_iter=max_iter,
                return_losses=True,
                poison_function=oracle_poison_function,
                theta_update_estimator=RGD_update_estimator,
                sampling_estimator=classification_sampling_estimator,
                sampling_estimator_kwargs={'mu_0': mu_0, 'mu_f': mu_f, 'sigma_0': sigma_0, 'sigma_1': sigma_1},
                poison_steps=poison_steps,
                poison_step_size=poison_step_size,
                epsilon=epsilon,
                delta=delta,
    )

    return all_theta_clean, all_theta_poisoned, all_losses_clean, all_losses_poisoned

all_theta_clean_list = []
all_theta_poisoned_list = []
all_loss_clean_list = []
all_loss_poisoned_list = []

for trial in range(n_trials):
    all_theta_clean, all_theta_poisoned, all_loss_clean, all_loss_poisoned = run_trial()
    all_theta_clean_list.append(all_theta_clean) 
    all_theta_poisoned_list.append(all_theta_poisoned)
    all_loss_clean_list.append(all_loss_clean)
    all_loss_poisoned_list.append(all_loss_poisoned)

fig, (ax1, ax2) = plot_results_2d(all_theta_clean_list, 
                    all_theta_poisoned_list,
                    all_loss_clean_list, 
                    all_loss_poisoned_list
                    )

plt.show()